In [ ]:
# the only package that this script requires is the random package
# otherwise, base python has everything we need
import random

# i did not use ChatGPT at all, and it's probably obvious

# here are objects which are globally defined

class Constants:
    SUITS = ["Spades", "Clubs", "Diamonds", "Hearts"] # the suits
    ART=["♠︎","♣︎","♦︎","♥︎"]
    num_cards_per_suit = 13 # italian deck
    VALUES = range(1, num_cards_per_suit+1) # number of cards (can be changed)
    SYMBOLS = ['A','2','3','4','5','6','7','8','9','10','J','Q','K'] # symbols associated with each card
    # POSITIONS = ["Hand","Tableau","Suitstacks"] # these are what I named each position of the board
    # the hand means the draw deck, the tableau means the table of cards in piles,
    # the suitstacks are where the suits are organized to win

    # here are where legal moves for the game are defined
    # it takes ``cartota`` ``playota`` and ``board`` as input
    # sorry for the weird and annoying names of the variables
    # the first two objects, ``cartota and ``playota`` are tuples
    # cartota = (Card, (position, extra position info)) and same formatting for ``playota``
    # the cartota is the card which is being played
    # and the playota is the card where playing on
    # the Card class is defined below
    # the position is a string, ^see above line POSITIONS
    # the extra info means where in that position our card is
    # for the Hand it is always None, for the Tableau, it says which pile we are in
    # for the Suitstacks it says which suit we are in
    # this function is called in the ``check_playable()`` method of the Board class (defined below)
    def legal_move(cartota, playota, board): # returns whether cartota can be played on playota, True or False

        card = cartota[0]
        location = cartota[1][0]
        loc_extra = cartota[1][1]

        playon_card = playota[0]
        playon_loc = playota[1][0]
        playon_extra = playota[1][1]


        if playon_loc == "Tableau": # if we are playing on the Tableau
            if playon_card.empty: # if that spot in the Tableau is empty
                return card.value == Constants().VALUES[-1] # only the King can go there
            
            # otherwise tell us if the cards are "braided" (see below)
            # meaning opposite color and descending value
            else: return card.is_braided_with(playon_card)
            
        else: # playon_loc == "Suitstacks": # we are playing on the suitstacks
            if playon_extra == card.suit: # the card must go in the appropriate suitstack, i.e. matching suits
                if playon_card.empty: return card.value == Constants().VALUES[0] # if the stack is empty, only the Ace goes there

                # if we are playing from the Tableau, double check that we are using an unblocked card
                # otherwise, the location is the hand
                elif location == "Tableau" and board.tableau.piles[loc_extra].face_up[-1] == card or location == "Hand":
                    return card.value - playon_card.value == 1 # the cards in the suitstacks must be in ascending order
                
                else: return False

            else: return False
    
    
           

# this is the Card class
# a non-empty card has a value and a suit
# otherwise, it is empty.  it is useful to define empty spaces for the solitaire rules^
class Card:
    def __init__(self, value=None, suit=None, empty=False):
        if empty==True:
            self.empty=True
            self.name = "Empty"
        else:
            self.suit = suit
            if self.suit == "Spades" or self.suit == "Clubs": self.color = "Black"
            else: self.color = "Red"
            self.value = value
            self.name = str(Constants().SYMBOLS[self.value-1])+" of "+self.suit # this makes it easy to identify cards in a print
            self.symbol=str(Constants().SYMBOLS[self.value-1])+Constants().ART[Constants().SUITS.index(self.suit)]
            self.empty=False
    
    # a "braid" is when two cards are separated by 1 in value and opposite colors
    def is_braided_with(self, playon_card): return playon_card.value - self.value == 1 and playon_card.color != self.color



# the deck class essentially tells us how to shuffle and deal
# if we want to specify some deck order, then we can do so.
class Deck:
    def __init__(self, desired_order=None):
        self.cards = []

        if desired_order is not None: self.cards = desired_order

        else: # just put the cards in order
            for suit in Constants().SUITS:
                for value in Constants().VALUES:
                    self.cards += [Card(value,suit)]
                      
    def shuffle(self): # then shuffle them
        random.shuffle(self.cards) # the ``shuffle`` method of ``random`` does exactly that

    # here is the particular way in which cards are dealt
    def deal(self):
        num_piles = 7 # 7 piles
        num_tableau = int(num_piles/2*(num_piles+1)) # 28 total in the tableau
        piles=[]
        for i in range(num_piles): # this loop creates the piles
            cards=self.cards[int(i/2*(i+1)):int((i+1)/2*(i+2))]
            if i==0: face_up,face_down = (cards,[]) # the first pile has no face down cards
            else: face_up,face_down=(cards[-1:],cards[:-1]) # otherwise only the last card is face up

            # the Pile class is defined below, it is initialized with face up cards, face down cards, and an index
            # which indicates which pile it is on the tableau
            piles += [Pile(face_up_cards=face_up, face_down_cards=face_down, pile_index=i)]

        # the Tableau class is defined below, it is initialized with a list of piles
        tableau = Tableau(piles=piles)

        # the Hand class is defined below, it is initialized with a list of cards
        hand = Hand(self.cards[num_tableau:]) # the rest of the cards not in the Tableau are put into the Hand

        # the Suitstacks class is defined below as well as the Stack class
        # Suitstacks are initialized with stacks and a Stack is initialized with a Suit and a potential list of cards
        # here, when the cards are dealt, all stacks are empty, but we can initialize a board with nonempty stacks
        suitstacks = Suitstacks(stacks=[Stack(suit=suite) for suite in Constants().SUITS])

        # the Board class is defined below, it is initialized with a Tableau, a Hand, and a Suitstacks
        # this is meant to represent a game of Solitaire
        return Board(tableau=tableau, hand=hand, suitstacks=suitstacks)
    
# each object: Tableau, Hand, and Suitstacks will have a method called ``available()``
# these will be called in the Board method ``check_available()`` telling us which cards are available to play
# ``available()`` returns a list of ``cartota`` objects (see above for cartota definition)
# out of all of the available cards, we then see if there exist any legal moves for that card using ``legal_move`` method of Constants

# Hand is initialized with a list of Card objects
class Hand:
    def __init__(self, cards_in_hand):
        self.cards = cards_in_hand
    
    # all cards in the hand are available to play
    # unless the hand has no cards
    def available(self):
        if len(self.cards) == 0: return []
        else: return [(card,("Hand",None)) for card in self.cards]
    
    # how to copy
    def copy(self): return Hand(cards_in_hand=self.cards.copy())
        
# Tableau is initialized with a list of Pile objects
class Tableau:
    def __init__(self, piles):
        self.piles = piles
    
    # all face up cards in the Tableau are available
    def available(self):
        available = []
        for pile in self.piles:
            if pile.is_empty(): continue
            
            for card in pile.face_up:
                if card.value == Constants().VALUES[-1] and len(pile.face_down) == 0: continue
                available += [(card,("Tableau",pile.pile_index))]
        return available

    # copy each pile, see below
    def copy(self): return Tableau(piles=[pile.copy() for pile in self.piles])


# Pile is initialized with a list of face down cards, a list of face up cards, and a pile index
# the card lists are lists of Card objects, and the pile_index is an integer from 0 to 6
class Pile:
    def __init__(self, face_down_cards, face_up_cards, pile_index):
        self.pile_index = pile_index
        self.face_up = face_up_cards
        self.face_down = face_down_cards
    
    # it's useful to know if a pile is empty
    def is_empty(self):
        # here ``cards`` is a method rather than an attribute
        if len(self.cards())==1: return True
    
    # i decided to do a method because we update the face up and face down cards
    # and we always want an ``empty`` Card to be at the front of the list
    # sometimes self.face_up = [] will be an empty list and likewise self.face_down
    def cards(self): return [Card(empty=True)] + self.face_down + self.face_up

    # copy the pile
    def copy(self): return Pile(face_down_cards=self.face_down.copy(),face_up_cards=self.face_up.copy(),pile_index=self.pile_index)

# Suitstacks is initialized with a list of Stack objects
class Suitstacks:
    def __init__(self, stacks):
        # this is a dictionary where the key is one of the strings in Constants().SUITS
        self.stacks = {stack.suit:stack for stack in stacks}

    # if the suitstack is not empty, then its available cards is only the one on top
    def available(self):
        available = []
        for suit in Constants().SUITS:
            stack = self.stacks[suit]
            if stack.is_empty(): continue
            available += [(stack.cards[-1],("Suitstacks",suit))]
        return available
    
    # copy each stack, see below
    def copy(self): return Suitstacks(stacks=[self.stacks[suit].copy() for suit in Constants().SUITS])

# a stack object is initialized with a suit (string from Constants().SUITS) and a list of Card objects
# by default, it is empty
class Stack:
    def __init__(self, suit, cards=[]):
        self.suit = suit
        self.cards = [Card(empty=True)] + cards

    # it's useful to know when the suitstack is empty
    def is_empty(self):
        if len(self.cards)==1: return True

    # copy the stack (remember to exclude the empty card)
    def copy(self): return Stack(suit=self.suit,cards=self.cards.copy()[1:])
    
# Board is initialized with a Hand object, a Tableau object, and a Suitstacks object
# each of these will correspond to attributes of the Board
class Board:
    def __init__(self, hand, tableau, suitstacks, store_history = False): # i have not implemented the history
        self.hand = hand
        self.tableau = tableau
        self.suitstacks = suitstacks
        self.no_moves = False

    # the script will evolve the solitaire game randomly
    def random_play(self):
        available = self.check_available() # check what's available (see below)
        while len(available)>0: # we filter cards which have no legal moves
            cartota = random.choice(available) # select a random available card (a cartota is a tuple, see above)
            playable = self.check_playable(cartota) # check what we can legally play on (see below)
            
            # if we can play
            if len(playable) != 0:
                # then choose a random move (see below for ``move`` method definition
                self.move(cartota_move=cartota, playota_onto=random.choice(playable))
                break # break the loop once we have moved

            # if there are no legal moves for this card, then remove it from the list
            else: available.remove(cartota)
    
    # this form of play will take a strategy as input
    # ``strategy`` is a tuple of functions: a heuristic and a priority function
    # the game will evolve based on which move has the highest heuristic value
    def greed_play(self, strategy):

        available = self.check_available()

        # this will be a list which contains all plays for every card in available
        # often the elements will be empty
        able_moves=[]

        # here is the heuristic and priority
        heuristic=strategy[0]
        priority=strategy[1]
        
        # h will be the heuristic for each play
        # so it will have the same dimension as all_plays
        h=[]
        # we will save which locations in ``h`` have the highest values
        outer_indices=[]
        inner_indices=[]
        best_scores=[] # this is a list of best heuristic scores for each element of ``available``

        for able in available:
            playable = self.check_playable(able)
            able_moves+=[playable]

            scores=[]
            if len(playable)==0: scores+=[-100] # make sure that we never have to look at cards which have no moves
            else: scores+=[heuristic(cartota_move=able,playota_onto=play,board=self) for play in playable] # evaluate heuristic
            h+=[scores] # save these scores in a list

            best_score=max(scores)
            best_scores+=[best_score]

            # these will tell us which plays have the highest heuristic
            # for a given card on the board
            # much of the time this value will just be -100
            inds=[]
            for i in range(len(scores)):
                if scores[i] == best_score: inds+=[i]
            outer_indices+=[inds]

        # this tells us which cards have the best heuristic
        best=max(best_scores)
        inner_indices=[]
        for j in range(len(best_scores)):
            if best_scores[j]==best: inner_indices+=[j]
        
        # multiple moves may have the highest heuristic value
        # this is often the case
        best_moves=[]
        for i in inner_indices:
            if able_moves[i]==[]: continue
            for j in outer_indices[i]:
                best_moves += [(available[i],able_moves[i][j])]
        
        if len(best_moves)==0: self.no_moves = True

        # if there is only one best move we can go ahead and play it
        elif len(best_moves)==1: self.move(cartota_move=best_moves[0][0],playota_onto=best_moves[0][1])

        # otherwise we break the tie using priorities
        else:
            p=[] # list of priorities
            p+=[priority(cartota_move=best_moves[i][0], playota_onto=best_moves[i][1], board=self) for i in range(len(best_moves))]

            best_p=max(p) # find the highest priority
            prio_moves=[]
            for i in range(len(p)):
                if p[i]==best_p: prio_moves+=[best_moves[i]]
            
            # if we have a tie among priorities, then choose randomly
            if len(prio_moves)==1: self.move(cartota_move=prio_moves[0][0],playota_onto=prio_moves[0][1])
            else:
                chosen_move = random.choice(prio_moves)
                self.move(cartota_move=chosen_move[0],playota_onto=chosen_move[1])
        
   
    def check_available(self): # look at all parts of the board and see what's available
        hand_available = self.hand.available()
        tableau_available = self.tableau.available()
        suitstacks_available = self.suitstacks.available()
        return hand_available + tableau_available + suitstacks_available # a big list of cartota tuples

    # for this method we pass a cartota, which is what i call a tuple of a Card object and its place on the board (see above)
    def check_playable(self, cartota):
        # look the last card in each pile (which includes empty spaces)
        # and look at the last card in each suistack (also includes empty spaces)
        # these are in general the kind of places we can play
        all_plays = []
        for pile in self.tableau.piles:
            all_plays += [(pile.cards()[-1], ("Tableau",pile.pile_index))]
        for suit in Constants().SUITS:
            stack = self.suitstacks.stacks[suit]
            all_plays += [(stack.cards[-1], ("Suitstacks",suit))]
        
        # a playota is the same as a cartota, a tuple of a Card object and its place on the board
        # i just call it playota so we know which card we are playing and which one is being played on
        dum=all_plays.copy()
        for play in all_plays:
            # return only playotas which the cartota can legally move to
            if not Constants.legal_move(cartota=cartota, playota=play, board=self): dum.remove(play)
        
        return dum
        
    # this is how cards move on boards
    # ``cartota_move`` is a cartota of the card we are moving
    # ``playota_onto`` is a cartota we are playing on (playota)
    # we already assume that the cards have been filtered in the ``check_playable`` method
    def move(self, cartota_move, playota_onto):
        card = cartota_move[0]
        location = cartota_move[1][0]
        loc_extra = cartota_move[1][1]

        # play = playota_onto[0] # turns out we don't need to know this
        play_loc = playota_onto[1][0]
        play_extra = playota_onto[1][1]

        if location == "Hand": # if we are playing from the hand
            self.hand.cards.remove(card) # remove the card from the hand

            if play_loc == "Tableau": # if we are playing onto the Tableau

                # then add that card to the front of the Tableau
                self.tableau.piles[play_extra].face_up += [card]

            else: # play_loc == "Suitstacks": # otherwise we must be playing on the suitstacks
                self.suitstacks.stacks[play_extra].cards += [card] # add the card to the top of the suitstack
        
        elif location == "Tableau": # otherwise if we are playing from the tableau

            # when a blocked card is moved from the tableau, we have to move all of the cards below it
            moving_pile = self.tableau.piles[loc_extra].face_up # these are all face up cards in the pile of interest
            position = moving_pile.index(card) # this tells us where our card of interest lies in the pile
            chunk = moving_pile[position:] # we move the card along with everything that follows
            self.tableau.piles[loc_extra].face_up = moving_pile[:position] # we remove all of these cards from the face up cards of that pile

            # sometimes we are moving an unblocked card from the tableau which is blocking a face down card
            if position == 0 and len(self.tableau.piles[loc_extra].face_down) > 0:
                flip = self.tableau.piles[loc_extra].face_down[-1] # this is the card we flip from face down to face up
                self.tableau.piles[loc_extra].face_down.remove(flip) # it is no longer face down
                self.tableau.piles[loc_extra].face_up = [flip] # it is now face up

            if play_loc == "Tableau": # if we are playing onto another card in the tableau
                self.tableau.piles[play_extra].face_up += chunk # then slap the chunk on that card
            
            else: # play_loc == "Suitstacks": # otherwise we must playing onto the suitstacks
                self.suitstacks.stacks[play_extra].cards += [card] # so we add that card to the stack
                # note that we have filtered inappropriate plays from tableau onto the suistacks
            
        else: # location == "Suistacks": # if we are not playing from the Hand or Tableau, then we must be playing from the suistacks
            self.suitstacks.stacks[loc_extra].cards.remove(card) # remove that card from the stack
            self.tableau.piles[play_extra].face_up += [card] # add that card to the pile (we can only play from the suitstacks to a tableau pile)

    
    def show(self): # print out which cards in what parts of the board
        # i wanted to implement some cool text art but did not have time
        print("no. cards in hand: "+str(len(self.hand.cards))+"\n")
        for pile in self.tableau.piles:
            st="pile: "
            st+="\n\tface down: "
            for card in pile.face_down: st+=card.name+","
            st+="\n\tface up: "
            for card in pile.face_up: st+=card.name+","
            print(st)
        st2="\nSuitstacks:\n\t"
        for suit in Constants().SUITS:
            st2+=suit+": "+str(len(self.suitstacks.stacks[suit].cards)-1)+"\t"
        print(st2)

    # how to copy the board
    def copy(self): return Board(hand=self.hand.copy(),tableau=self.tableau.copy(),suitstacks=self.suitstacks.copy())

    def check_win(self): return all([len(pile.face_down)==0 for pile in self.tableau.piles])

    # def check_loss(self): 

# program the heuristic:

# my strategy is to flip as many cards up as quick as possible
# since once all cards are face-up, the game is more or less trivially won
def nich_heuristic(cartota_move, playota_onto, board):
    card = cartota_move[0]
    location = cartota_move[1][0]
    loc_extra = cartota_move[1][1]

    # play = playota_onto[0]
    play_loc = playota_onto[1][0]
    # play_extra = playota_onto[1][1]

    # if we play from the tableau
    if location == "Tableau":
        position = board.tableau.piles[loc_extra].face_up.index(card)
        num_facedown = len(board.tableau.piles[loc_extra].face_down)
        # if we can flip a face down card from the pile then we should do it
        # and we do so based on how many face down cards are in the pile;
        # we flip larger piles first
        if position == 0: return 5 + num_facedown
    
    # in general we want cards to go into suitstacks
    if play_loc == "Suitstacks": return 5

    # other scenarios are tied
    return 0

def nich_priority(cartota_move, playota_onto, board):#, playota_onto):
    card = cartota_move[0]
    location = cartota_move[1][0]
    # loc_extra = cartota_move[1][1]

    # play = playota_onto[0]
    play_loc = playota_onto[1][0]
    # play_extra = playota_onto[1][1]

    

    # we want to play our highest-ranking card last
    # and we want to withold it if nothing can be played on it
    # otherwise it's good to play
    if location == "Hand":
        if card.value == Constants().VALUES[-1]:
            available = board.check_available()
            for able in available:
                if able[0].is_braided_with(card): return 1
            return -1
        
    # IN GENERAL if we can play onto the Tableau from Hand or Suitstacks
    # then out of all of our plays we should prioritize those which flip cards
    if play_loc == "Tableau" and location != "Tableau":
        for pile in board.tableau.piles:
            if pile.is_empty(): continue
            top_card = pile.face_up[0]
            num_facedown = len(pile.face_down)
            if top_card.is_braided_with(card): return 1 + num_facedown
    
    # we generally want to keep suitstack cards there
    # unless putting it on the tableau can flip another card
    if location == "Suitstacks": return -5
    
    # any other scenarios are equally 0
    return 0

nich_strategy = (nich_heuristic,nich_priority)


# strategy taken from Yan et al 2005

def yan_heuristic(cartota_move, playota_onto, board):
    # card = cartota_move[0]
    location = cartota_move[1][0]
    # loc_extra = cartota_move[1][1]

    # play = playota_onto[0]
    play_loc = playota_onto[1][0]
    # play_extra = playota_onto[1][1]

    if play_loc == "Suitstacks": return 5

    elif location == "Hand" and play_loc == "Tableau": return 5

    elif location == "Suitstacks" and play_loc == "Tableau": return -10

    else: return 0

def yan_priority(cartota_move, playota_onto, board):
    card = cartota_move[0]
    location = cartota_move[1][0]
    loc_extra = cartota_move[1][1]

    # play = playota_onto[0]
    play_loc = playota_onto[1][0]
    # play_extra = playota_onto[1][1]

    if play_loc == "Tableau":

        if location == "Tableau":
            position = board.tableau.piles[loc_extra].face_up.index(card)
            num_facedown = len(board.tableau.piles[loc_extra].face_down)
            if position == 0 and num_facedown > 0: return len(board.tableau.piles[loc_extra].face_down)+1
        
        elif location == "Hand":
            if card.value == Constants().VALUES[-1]:
                for able in board.check_available():
                    if able[0].is_braided_with(card): return 1
                
                return -1

            else: return 1
    
    return 0

yan_strategy = (yan_heuristic,yan_priority)